# Day 73: sklearn Pipeline — Production-Ready ML

## Objective
- Chain preprocessing and a model into one `Pipeline`
- Use `ColumnTransformer` to treat numeric and categorical columns differently
- Tune everything together with `GridSearchCV`, with no leakage
- Go from raw data to predictions with one object (Titanic dataset)

> **Data note:** `dataset.csv` is the real Titanic dataset (891 passengers, Kaggle format).
> The model predicts `Survived` (1 = survived).

## Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

## Theory
- A **Pipeline** is a chain: each step transforms the data, and the last step is the model
- `fit` on a pipeline fits every step on the **training data only**, in order
- `predict` on new data runs the same steps automatically
- **Data leakage** = test-set information sneaking into training (for example, scaling with the mean of ALL rows)
- Pipelines + cross-validation refit every step inside each fold, so leakage is not possible
- **ColumnTransformer** applies different steps to different columns, in one object

In [2]:
df = pd.read_csv("dataset.csv")

numeric_cols = ["Age", "SibSp", "Parch", "Fare"]
categorical_cols = ["Pclass", "Sex", "Embarked"]

X = df[numeric_cols + categorical_cols]
y = df["Survived"]

print(X.shape)
print(X.isna().sum()[X.isna().sum() > 0])

(891, 7)
Age         177
Embarked      2
dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (712, 7) | Test: (179, 7)


## Example 1: A simple Pipeline (numeric columns only)

In [4]:
simple = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
])
simple.fit(X_train[numeric_cols], y_train)

pred = simple.predict(X_test[numeric_cols])
print("Test accuracy:", round(accuracy_score(y_test, pred), 3))
print("Steps:", [name for name, step in simple.steps])

Test accuracy: 0.575
Steps: ['imputer', 'scaler', 'model']


## Example 2: Why leakage matters

In [5]:
# WRONG: the scaler learns from ALL rows, including future test rows
scaler_all = StandardScaler().fit(X[["Fare"]])

# RIGHT: the scaler learns from training rows only
scaler_train = StandardScaler().fit(X_train[["Fare"]])

print("Fare mean (all rows):  ", round(scaler_all.mean_[0], 3))
print("Fare mean (train only):", round(scaler_train.mean_[0], 3))
print("Difference:", round(abs(scaler_all.mean_[0] - scaler_train.mean_[0]), 3))

Fare mean (all rows):   32.204
Fare mean (train only): 31.82
Difference: 0.384


In [6]:
# Leaky vs leak-free cross-validation (5 folds, AUC)
X_leaky = StandardScaler().fit_transform(X[numeric_cols].fillna(X[numeric_cols].median()))
rf = RandomForestClassifier(n_estimators=100, random_state=42)

leaky = cross_val_score(rf, X_leaky, y, cv=5, scoring="roc_auc")
safe = cross_val_score(simple, X[numeric_cols], y, cv=5, scoring="roc_auc")

print("Leaky CV AUC:     ", round(leaky.mean(), 3))
print("Pipeline CV AUC:  ", round(safe.mean(), 3))

Leaky CV AUC:      0.725
Pipeline CV AUC:   0.723


## Example 3: ColumnTransformer (numeric + categorical)

In [7]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols),
])

X_prepared = preprocessor.fit_transform(X_train)
print("Before:", X_train.shape)
print("After: ", X_prepared.shape)
print(preprocessor.get_feature_names_out())

Before: (712, 7)
After:  (712, 12)
['num__Age' 'num__SibSp' 'num__Parch' 'num__Fare' 'cat__Pclass_1'
 'cat__Pclass_2' 'cat__Pclass_3' 'cat__Sex_female' 'cat__Sex_male'
 'cat__Embarked_C' 'cat__Embarked_Q' 'cat__Embarked_S']


## Example 4: The full pipeline, raw data to predictions

In [8]:
full = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42)),
])

cv_auc = cross_val_score(full, X_train, y_train, cv=5, scoring="roc_auc")
print("CV AUC:", round(cv_auc.mean(), 3), "+/-", round(cv_auc.std(), 3))

full.fit(X_train, y_train)
probs = full.predict_proba(X_test)[:, 1]
print("Test AUC:     ", round(roc_auc_score(y_test, probs), 3))
print("Test accuracy:", round(accuracy_score(y_test, full.predict(X_test)), 3))

CV AUC: 0.853 +/- 0.036


Test AUC:      0.831
Test accuracy: 0.816


## Example 5: Pipeline inside GridSearchCV

In [9]:
param_grid = {
    "prep__num__imputer__strategy": ["median", "mean"],
    "model__n_estimators": [100, 300],
    "model__max_depth": [3, 5, None],
}

grid = GridSearchCV(full, param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:")
for name, value in grid.best_params_.items():
    print("  ", name, "=", value)
print("Best CV AUC:", round(grid.best_score_, 3))
print("Test AUC:   ", round(roc_auc_score(y_test, grid.predict_proba(X_test)[:, 1]), 3))

Best params:
   model__max_depth = 5
   model__n_estimators = 300
   prep__num__imputer__strategy = mean
Best CV AUC: 0.872
Test AUC:    0.847


## Example 6: Predict from raw data

In [10]:
# A brand-new passenger, exactly as raw data (no manual scaling or encoding)
new_passenger = pd.DataFrame([{
    "Age": 28, "SibSp": 0, "Parch": 0, "Fare": 80.0,
    "Pclass": 1, "Sex": "female", "Embarked": "C",
}])
print("Survival probability:", round(grid.predict_proba(new_passenger)[0, 1], 3))

# Missing age and a port we never saw in training: the pipeline still works
odd = new_passenger.copy()
odd["Age"] = np.nan
odd["Embarked"] = "X"
print("With missing Age and unknown port:", round(grid.predict_proba(odd)[0, 1], 3))

Survival probability: 0.974
With missing Age and unknown port: 0.97


## Example 7: Save and reload the whole pipeline

In [11]:
import joblib, os

joblib.dump(grid.best_estimator_, "titanic_pipeline.joblib")
reloaded = joblib.load("titanic_pipeline.joblib")

same = np.allclose(reloaded.predict_proba(X_test), grid.predict_proba(X_test))
print("Reloaded pipeline gives identical predictions:", same)
print("File size (KB):", round(os.path.getsize("titanic_pipeline.joblib") / 1024))

Reloaded pipeline gives identical predictions:

 True
File size (KB): 1226


## Practice Exercise
Swap the model: replace the Random Forest with `LogisticRegression` in the same pipeline. Which has the better CV AUC?

In [12]:
logit_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])
logit_auc = cross_val_score(logit_pipe, X_train, y_train, cv=5, scoring="roc_auc")
print("Logistic CV AUC:", round(logit_auc.mean(), 3), "+/-", round(logit_auc.std(), 3))
print("Forest CV AUC:  ", round(cv_auc.mean(), 3), "+/-", round(cv_auc.std(), 3))

Logistic CV AUC: 0.855 +/- 0.023
Forest CV AUC:   0.853 +/- 0.036


## Mini Challenge
Add a `FamilySize` feature (`SibSp + Parch + 1`) **inside** the workflow and see if test AUC changes.

In [13]:
# Keep it leak-free: build the new column in the data, then use the same pipeline
X_fam = X.copy()
X_fam["FamilySize"] = X_fam["SibSp"] + X_fam["Parch"] + 1

Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_fam, y, test_size=0.2, random_state=42, stratify=y
)
prep_fam = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols + ["FamilySize"]),
    ("cat", categorical_pipe, categorical_cols),
])
fam_pipe = Pipeline([
    ("prep", prep_fam),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42)),
]).fit(Xf_train, yf_train)

print("Test AUC without FamilySize:", round(roc_auc_score(y_test, probs), 3))
print("Test AUC with FamilySize:   ", round(roc_auc_score(yf_test, fam_pipe.predict_proba(Xf_test)[:, 1]), 3))

Test AUC without FamilySize: 0.831
Test AUC with FamilySize:    0.828


## Summary
- A Pipeline chains preprocessing and a model, and fits every step on training data only
- ColumnTransformer handles numeric and categorical columns in one object
- Putting the pipeline inside GridSearchCV tunes everything together without leakage
- One saved pipeline takes raw data to predictions, even with missing values or new categories
- Swapping models is a one-line change